In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from loguru import logger
import os

print("Current working directory:", os.getcwd())

# Configure loguru for notebook display
# logger.remove()
# logger.add(lambda msg: print(msg, end=''), colorize=True, format="{level.icon} {message}")

# Data paths
ANNOT_FILE = "data/dataset_092624.xlsx"
OUTPUT_FILE = "data/dataset_092624_validated.xlsx"

Current working directory: /home/user/llm_metadata


## 1. Load Raw Data

In [3]:
# Load the raw Excel file
raw_df = pd.read_excel(ANNOT_FILE)

print(f"Loaded {len(raw_df)} rows, {len(raw_df.columns)} columns")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nSource distribution:")
print(raw_df['source'].value_counts())
print(f"\nTotal valid (valid_yn='yes'): {(raw_df['valid_yn'] == 'yes').sum()}")
raw_df.head()

Loaded 418 rows, 43 columns

Columns: ['id', 'url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_articles', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'url.1', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']

Source distribution:
source
semantic_scholar    254
zenodo              114
dryad                47
referenced            3
Name: count, dtype: int64

,id,url,title,full_text,publication_year,source,id_query,reason_non_valid,valid_yn,dataset_relevance,...,threatened_species,new_species_science,new_species_region,bias_north_south,url.1,MC_dataset_type,MC_spatial_range,MC_temporal_range,MC_relevance,MC_relevance_modifiers
0,3,https://doi.org/10.5061/dryad.05qfttf0n,Fish mock community with 41 species from 13 or...,Fish mock community with 41 species from 13 or...,2020,zenodo,3,NaN,yes,X,...,NaN,NaN,NaN,NaN,https://doi.org/10.5061/dryad.05qfttf0n,L,X,X,X,X
1,4,https://doi.org/10.5061/dryad.0rxwdbs2c,Exploration and diet specialization in eastern...,Exploration and diet specialization in eastern...,2022,zenodo,"3,7,4,6",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.0rxwdbs2c,L,L,M,L,L
2,5,https://doi.org/10.5061/dryad.121sk,Data from: Paternity analysis of wood turtles ...,Data from: Paternity analysis of wood turtles ...,2017,zenodo,"3,8",NaN,yes,M,...,True,False,False,False,https://doi.org/10.5061/dryad.121sk,H,X,L,L,M
3,7,https://doi.org/10.5061/dryad.12jm63xtw,Data from: 60 specific eDNA qPCR assays to det...,Data from: 60 specific eDNA qPCR assays to det...,2020,zenodo,"3,8",NaN,yes,X,...,NaN,NaN,NaN,NaN,https://doi.org/10.5061/dryad.12jm63xtw,X,X,X,X,X
4,9,https://doi.org/10.5061/dryad.1771t,Data from: Resampling method for applying dens...,Data from: Resampling method for applying dens...,2016,dryad,"0,3,4,8,10",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.1771t,H,X,L,L,L


## 1.5 Inspect URL Fields for SS Records

In [4]:
import re

# Separate records by source
ss_df = raw_df[raw_df['source'] == 'semantic_scholar'].copy()
dryad_zenodo_df = raw_df[raw_df['source'].isin(['dryad', 'zenodo'])].copy()

print(f"Semantic Scholar records: {len(ss_df)}")
print(f"Dryad/Zenodo records:     {len(dryad_zenodo_df)}")

# Inspect url vs url.1 for SS records
url_same = (ss_df['url'] == ss_df['url.1']).sum()
url_both_null = (ss_df['url'].isna() & ss_df['url.1'].isna()).sum()
url_diff = len(ss_df) - url_same - url_both_null

print(f"\n--- SS URL column comparison ---")
print(f"url == url.1 (identical): {url_same}")
print(f"Both null:                {url_both_null}")
print(f"Different:                {url_diff}")

# Show sample
print(f"\n--- SS url sample (first 10) ---")
print(ss_df[['url', 'url.1']].head(10).to_string())

print(f"\n--- Dryad/Zenodo url sample (first 5) ---")
print(dryad_zenodo_df[['url', 'url.1']].head(5).to_string())

Semantic Scholar records: 254
Dryad/Zenodo records:     161

--- SS URL column comparison ---
url == url.1 (identical): 224
Both null:                28
Different:                2

--- SS url sample (first 10) ---
                                                   url                                              url.1
104                 https://doi.org/10.5558/TFC76653-4                 https://doi.org/10.5558/TFC76653-4
105                    https://doi.org/10.1139/B86-122                    https://doi.org/10.1139/B86-122
106                                                NaN                                                NaN
107                    https://doi.org/10.1139/X88-250                    https://doi.org/10.1139/X88-250
108          https://doi.org/10.1023/A%3A1011871723686          https://doi.org/10.1023/A%3A1011871723686
109      https://doi.org/10.1080/11956860.2021.1885804      https://doi.org/10.1080/11956860.2021.1885804
110  https://doi.org/10.1078/S0031-4056%280

In [5]:
## 1.6 Column Rename: url → source_url, cited_articles → cited_article_doi
#
# Rename BEFORE validation so DataFrameValidator maps to schema field names.
# The raw xlsx is untouched; this is in-memory only.
# url.1 is confirmed identical to url for all SS records — drop it.

COLUMN_RENAME = {'url': 'source_url', 'cited_articles': 'cited_article_doi'}
raw_df = raw_df.rename(columns=COLUMN_RENAME)
if 'url.1' in raw_df.columns:
    raw_df = raw_df.drop(columns=['url.1'])

print(f"Renamed columns: {list(COLUMN_RENAME.keys())} → {list(COLUMN_RENAME.values())}")
print(f"Dropped: url.1")
print(f"Columns now: {list(raw_df.columns)}")

Renamed columns: ['url', 'cited_articles'] → ['source_url', 'cited_article_doi']
Dropped: url.1
Columns now: ['id', 'source_url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_article_doi', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']


## 2. Run Validation

In [6]:
from llm_metadata.schemas import DataFrameValidator, ValidationReport
from llm_metadata.schemas.fuster_features import DatasetFeaturesNormalized

# Validate ALL records (Dryad + Zenodo + SS) through updated DatasetFeaturesNormalized
# This includes WU-A1 modulator fields: time_series, multispecies, threatened_species,
# new_species_science, new_species_region, bias_north_south
validator = DataFrameValidator(model=DatasetFeaturesNormalized, strict=False)
report = validator.validate(raw_df)

print(report.summary())

# Source-level breakdown
print("\nBreakdown by source:")
for source in raw_df['source'].value_counts().index:
    source_idx = set(raw_df[raw_df['source'] == source].index)
    n_valid = len([i for i in report.valid_indices if i in source_idx])
    n_invalid = len([i for i in report.invalid_indices if i in source_idx])
    print(f"  {source:<20}: {n_valid} valid, {n_invalid} invalid")

2026-02-18 05:07:16.881 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 418 valid, 0 invalid, 0 errors


VALIDATION REPORT
Total rows:     418
Valid rows:     418 (100.0%)
Invalid rows:   0

Error Breakdown:
  Type/Schema errors: 0
  Data/Value errors:  0

Errors by Field:

Breakdown by source:
  semantic_scholar    : 254 valid, 0 invalid
  zenodo              : 114 valid, 0 invalid
  dryad               : 47 valid, 0 invalid
  referenced          : 3 valid, 0 invalid


## 3. Review Errors

In [7]:
# Get errors as DataFrame for inspection
errors_df = report.errors_to_dataframe()

if len(errors_df) > 0:
    print(f"Total errors: {len(errors_df)}")
    print(f"\nError distribution by field:")
    print(errors_df['field'].value_counts())
    print(f"\nError distribution by type:")
    print(errors_df['error_type'].value_counts())
else:
    print("✅ No errors found!")

errors_df.head(20)

✅ No errors found!


,row_index,field,raw_value,error_type,message,suggested_fix


In [8]:
# View errors with suggested fixes
if len(errors_df) > 0:
    errors_with_fixes = errors_df[errors_df['suggested_fix'] != ''][['row_index', 'field', 'raw_value', 'suggested_fix']]
    print(f"Errors with suggested fixes: {len(errors_with_fixes)}")
    display(errors_with_fixes)

## 4. Review Invalid Rows

In [9]:
# Get invalid rows for correction
invalid_df = report.invalid_rows_to_dataframe()

# Select only relevant columns for display
relevant_cols = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
                 'temporal_range', 'temp_range_i', 'temp_range_f', 'taxons', 'referred_dataset']
available_cols = [c for c in relevant_cols if c in invalid_df.columns]

print(f"Invalid rows: {len(invalid_df)}")
if len(invalid_df) > 0:
    display(invalid_df[available_cols].head(10))

Invalid rows: 0


In [10]:
# --- SPECIES VALIDATION CHECK ---
print("Checking species field transformation...")

# Re-run validation with DatasetFeaturesNormalized to get validated rows
validator = DataFrameValidator(model=DatasetFeaturesNormalized, strict=False)
report = validator.validate(raw_df)
validated_df = report.valid_rows_to_dataframe()

# Rows that have species info
params_with_species = raw_df[raw_df['species'].notna()].index
print(f"Rows with species: {len(params_with_species)}")

if len(params_with_species) > 0:
    sample_indices = params_with_species[:50]

    print("\nComparing Raw vs Validated Species (First 50 examples):")
    comparison = pd.DataFrame({
        'Raw Species': raw_df.loc[sample_indices, 'species'],
        'Validated Species': validated_df.reindex(raw_df.index).loc[sample_indices, 'species']
    })

    # Check type of validated species
    valid_types = comparison['Validated Species'].apply(type).unique()
    print(f"Validated column types: {valid_types}")

    display(comparison)
else:
    print("No species data found in raw dataframe.")

2026-02-18 05:07:18.210 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 418 valid, 0 invalid, 0 errors


Checking species field transformation...
Rows with species: 244

Comparing Raw vs Validated Species (First 50 examples):
Validated column types: [<class 'list'>]


,Raw Species,Validated Species
0,41 fish mock species,[41 fish mock species]
1,Tamias striatus,[Tamias striatus]
2,Glyptemys insculpta,[Glyptemys insculpta]
3,40 freshwater vertebrate and invertebrates,[40 freshwater vertebrate and invertebrates]
4,"raccoons, striped skunks","[raccoons, striped skunks]"
5,white-tailed deer,[white-tailed deer]
6,Rangifer tarandus caribou,[Rangifer tarandus caribou]
7,Rangifer tarandus caribou,[Rangifer tarandus caribou]
8,Aspicilia bicensis,[Aspicilia bicensis]
9,Salvelinus namaycush,[Salvelinus namaycush]


## 5. Coverage Stats by Source

In [11]:
def has_doi(url):
    """Return True if the URL contains a DOI."""
    if pd.isna(url) or not url:
        return False
    return bool(re.search(r'doi\.org/', str(url), re.IGNORECASE))


sources_to_check = ['dryad', 'zenodo', 'semantic_scholar', 'referenced']
stats_rows = []

for source in sources_to_check:
    src_df = raw_df[raw_df['source'] == source]
    if len(src_df) == 0:
        continue
    n_total = len(src_df)
    n_valid = int((src_df['valid_yn'] == 'yes').sum())
    n_abstract = int(src_df['full_text'].notna().sum())
    n_doi = int(src_df['source_url'].apply(has_doi).sum())
    n_cited = int(src_df['cited_article_doi'].notna().sum())
    stats_rows.append({
        'source': source,
        'total_records': n_total,
        'valid_records': n_valid,
        'has_abstract': n_abstract,
        'abstract_pct': f"{100*n_abstract/n_total:.1f}%",
        'has_doi': n_doi,
        'doi_pct': f"{100*n_doi/n_total:.1f}%",
        'has_cited_articles': n_cited,
    })

coverage_df = pd.DataFrame(stats_rows)
print("Coverage Stats by Source:")
display(coverage_df)

print(f"\nTotals (all sources):")
print(f"  Total records:          {len(raw_df)}")
print(f"  Valid (valid_yn='yes'): {(raw_df['valid_yn'] == 'yes').sum()}")
print(f"  Has abstract:           {raw_df['full_text'].notna().sum()}")

Coverage Stats by Source:


,source,total_records,valid_records,has_abstract,abstract_pct,has_doi,doi_pct,has_cited_articles
0,dryad,47,37,46,97.9%,47,100.0%,44
1,zenodo,114,67,114,100.0%,114,100.0%,59
2,semantic_scholar,254,192,246,96.9%,156,61.4%,0
3,referenced,3,3,0,0.0%,0,0.0%,0



Totals (all sources):
  Total records:          418
  Valid (valid_yn='yes'): 299
  Has abstract:           406


## 6. Filter to Valid Records

In [12]:
# Filter schema-validated rows to those where valid_yn == 'yes'
# (biodiversity-relevant records only)
schema_valid_df = report.valid_rows_to_dataframe()
valid_records_df = schema_valid_df[schema_valid_df['valid_yn'] == 'yes'].copy()

print(f"Schema-valid rows:             {len(schema_valid_df)}")
print(f"Schema-valid + valid_yn='yes': {len(valid_records_df)}")

print(f"\nValid records by source:")
print(valid_records_df['source'].value_counts())

Schema-valid rows:             418
Schema-valid + valid_yn='yes': 299

Valid records by source:
source
semantic_scholar    192
zenodo               67
dryad                37
referenced            3
Name: count, dtype: int64


In [13]:
## Step A3.3: Fill cited_article_doi for records where it is missing
#
# SS records: source_url IS the article DOI (no separate cited_articles column)
# Dryad/Zenodo: use API fallback for records not covered by cited_article_doi column
#
# Note: valid_records_df.index holds original raw_df row indices, so we can look up
# non-schema columns (like 'id', 'title') from raw_df using that index.

from llm_metadata.article_retrieval import (
    extract_doi_from_url,
    get_article_doi_from_dryad_api,
    get_article_doi_from_zenodo_api,
)

# Add non-schema columns from raw_df (preserved via shared index)
for col in ['id', 'title']:
    if col in raw_df.columns:
        valid_records_df[col] = raw_df.loc[valid_records_df.index, col].values

# --- SS: extract DOI from source_url ---
ss_mask = (valid_records_df['source'] == 'semantic_scholar') & valid_records_df['cited_article_doi'].isna()
valid_records_df.loc[ss_mask, 'cited_article_doi'] = (
    valid_records_df.loc[ss_mask, 'source_url'].apply(extract_doi_from_url)
)
print(f"SS records filled: {ss_mask.sum()}")

# --- Dryad/Zenodo: API fallback for missing DOIs ---
dz_missing_mask = (
    valid_records_df['source'].isin(['dryad', 'zenodo']) &
    valid_records_df['cited_article_doi'].isna()
)
print(f"Dryad/Zenodo missing cited_article_doi: {dz_missing_mask.sum()} — querying APIs...")

for idx, row in valid_records_df[dz_missing_mask].iterrows():
    doi = None
    if row['source'] == 'dryad':
        doi = get_article_doi_from_dryad_api(row['source_url'])
    elif row['source'] == 'zenodo':
        doi = get_article_doi_from_zenodo_api(row['source_url'])
    if doi:
        valid_records_df.at[idx, 'cited_article_doi'] = doi

# Coverage summary
n_total = len(valid_records_df)
n_with_doi = valid_records_df['cited_article_doi'].notna().sum()
print(f"\ncited_article_doi coverage: {n_with_doi}/{n_total} ({100*n_with_doi/n_total:.1f}%)")
print(valid_records_df.groupby('source')['cited_article_doi'].apply(lambda s: s.notna().sum()).rename('with_doi').to_string())

SS records filled: 192
Dryad/Zenodo missing cited_article_doi: 31 — querying APIs...



cited_article_doi coverage: 250/299 (83.6%)
source
dryad                37
referenced            0
semantic_scholar    175
zenodo               38


In [14]:
## Step A3.4: Enrich journal_url, pdf_url, is_oa via OpenAlex (+ SS fallback)
#
# For all records with a cited_article_doi, query OpenAlex for OA metadata.
# Semantic Scholar is used as fallback for pdf_url when OpenAlex has none.

from llm_metadata.article_retrieval import enrich_article_metadata

# Ensure columns exist
for col in ['journal_url', 'pdf_url', 'is_oa']:
    if col not in valid_records_df.columns:
        valid_records_df[col] = None

enrichable_mask = valid_records_df['cited_article_doi'].notna()
enrichable_df = valid_records_df[enrichable_mask]
print(f"Records to enrich: {len(enrichable_df)}")

for idx, row in enrichable_df.iterrows():
    meta = enrich_article_metadata(row['cited_article_doi'])
    valid_records_df.at[idx, 'journal_url'] = meta['journal_url']
    valid_records_df.at[idx, 'pdf_url'] = meta['pdf_url']
    valid_records_df.at[idx, 'is_oa'] = meta['is_oa']

# Coverage summary
print("\n=== ENRICHMENT COVERAGE ===")
for col in ['cited_article_doi', 'journal_url', 'pdf_url', 'is_oa']:
    n = valid_records_df[col].notna().sum()
    total = len(valid_records_df)
    print(f"  {col:<22}: {n:>4}/{total} ({100*n/total:.1f}%)")

print("\nBy source:")
for source in ['dryad', 'zenodo', 'semantic_scholar']:
    src = valid_records_df[valid_records_df['source'] == source]
    n_oa = src['is_oa'].notna().sum()
    n_pdf = src['pdf_url'].notna().sum()
    print(f"  {source:<22}: is_oa={n_oa}/{len(src)}, pdf_url={n_pdf}/{len(src)}")

Records to enrich: 250



=== ENRICHMENT COVERAGE ===
  cited_article_doi     :  250/299 (83.6%)
  journal_url           :   98/299 (32.8%)
  pdf_url               :  189/299 (63.2%)
  is_oa                 :  194/299 (64.9%)

By source:
  dryad                 : is_oa=36/37, pdf_url=34/37
  zenodo                : is_oa=36/67, pdf_url=33/67
  semantic_scholar      : is_oa=122/192, pdf_url=122/192


In [15]:
## Step A3.5: Update dataset_article_mapping.csv with enriched columns + SS rows
#
# Additive strategy:
# - Original 6 columns unchanged (backward compatible with download notebooks)
# - Add journal_url, pdf_url, is_oa columns to existing rows
# - Add SS records (new rows) that weren't in the original Dryad/Zenodo mapping
# - Update article_doi for any row where API/extraction found a DOI

import os

MAPPING_CSV = "data/dataset_article_mapping.csv"

# Build enrichment lookup from valid_records_df
enrich_lookup = valid_records_df[['id', 'source_url', 'source', 'cited_article_doi', 'journal_url', 'pdf_url', 'is_oa']].copy()
enrich_lookup = enrich_lookup.rename(columns={
    'source_url': 'dataset_url',
    'cited_article_doi': 'article_doi_enriched',
})
# dataset_doi from source_url
enrich_lookup['dataset_doi'] = enrich_lookup['dataset_url'].apply(extract_doi_from_url)

if os.path.exists(MAPPING_CSV):
    mapping_df = pd.read_csv(MAPPING_CSV)
    print(f"Loaded existing mapping: {len(mapping_df)} rows, columns: {list(mapping_df.columns)}")
else:
    mapping_df = pd.DataFrame(columns=['id', 'dataset_doi', 'article_doi', 'source', 'retrieval_method', 'title'])
    print("No existing mapping CSV found; creating from scratch.")

# 1. Add new columns to existing rows via left-join on id
mapping_df = mapping_df.drop(columns=[c for c in ['journal_url', 'pdf_url', 'is_oa'] if c in mapping_df.columns])
mapping_df = mapping_df.merge(
    enrich_lookup[['id', 'journal_url', 'pdf_url', 'is_oa', 'article_doi_enriched']],
    on='id', how='left'
)

# 2. Fill article_doi where it was previously missing (API found it)
if 'article_doi' in mapping_df.columns:
    mapping_df['article_doi'] = mapping_df.apply(
        lambda r: r['article_doi'] if pd.notna(r['article_doi']) else r.get('article_doi_enriched'),
        axis=1
    )
mapping_df = mapping_df.drop(columns=['article_doi_enriched'], errors='ignore')

# 3. Add new SS rows not already in the mapping
existing_ids = set(mapping_df['id'].dropna().astype(int).tolist())
ss_new = enrich_lookup[
    enrich_lookup['source'] == 'semantic_scholar'
]
ss_new = ss_new[~ss_new['id'].isin(existing_ids)].copy()
if len(ss_new) > 0:
    ss_rows = ss_new[['id', 'dataset_doi', 'article_doi_enriched', 'source', 'journal_url', 'pdf_url', 'is_oa']].copy()
    ss_rows = ss_rows.rename(columns={'article_doi_enriched': 'article_doi'})
    ss_rows['retrieval_method'] = 'source_url'
    ss_rows['title'] = raw_df.loc[
        valid_records_df[valid_records_df['source'] == 'semantic_scholar'].index, 'title'
    ].values[:len(ss_rows)]
    mapping_df = pd.concat([mapping_df, ss_rows], ignore_index=True)
    print(f"Added {len(ss_rows)} new SS rows to mapping")
else:
    print("No new SS rows to add (already present or no SS records)")

mapping_df.to_csv(MAPPING_CSV, index=False)
print(f"\nSaved updated mapping to {MAPPING_CSV}")
print(f"Total rows: {len(mapping_df)}")
print(f"Columns: {list(mapping_df.columns)}")
n_pdf = mapping_df['pdf_url'].notna().sum()
n_oa = mapping_df['is_oa'].notna().sum()
n_doi = mapping_df['article_doi'].notna().sum()
print(f"\nCoverage:")
print(f"  article_doi: {n_doi}/{len(mapping_df)} ({100*n_doi/len(mapping_df):.1f}%)")
print(f"  pdf_url:     {n_pdf}/{len(mapping_df)} ({100*n_pdf/len(mapping_df):.1f}%)")
print(f"  is_oa:       {n_oa}/{len(mapping_df)} ({100*n_oa/len(mapping_df):.1f}%)")

Loaded existing mapping: 299 rows, columns: ['id', 'dataset_doi', 'article_doi', 'source', 'retrieval_method', 'title']
No new SS rows to add (already present or no SS records)

Saved updated mapping to data/dataset_article_mapping.csv
Total rows: 301
Columns: ['id', 'dataset_doi', 'article_doi', 'source', 'retrieval_method', 'title', 'journal_url', 'pdf_url', 'is_oa']

Coverage:
  article_doi: 252/301 (83.7%)
  pdf_url:     191/301 (63.5%)
  is_oa:       196/301 (65.1%)


## 7. Stats Table for Presentation

In [16]:
print("=== DATASET COVERAGE STATS FOR PRESENTATION ===\n")
header = f"{'Source':<22} {'Records':>9} {'Valid':>7} {'Abstract':>10} {'Has DOI':>9} {'Cited Art.':>11}"
sep = "-" * len(header)
print(header)
print(sep)

presentation_sources = ['dryad', 'zenodo', 'semantic_scholar']
totals = {'total': 0, 'valid': 0, 'abstract': 0, 'doi': 0, 'cited': 0}

for source in presentation_sources:
    src_df = raw_df[raw_df['source'] == source]
    n_total = len(src_df)
    n_valid = int((src_df['valid_yn'] == 'yes').sum())
    n_abstract = int(src_df['full_text'].notna().sum())
    n_doi = int(src_df['source_url'].apply(has_doi).sum())
    n_cited = int(src_df['cited_article_doi'].notna().sum())
    print(f"{source:<22} {n_total:>9} {n_valid:>7} {n_abstract:>10} {n_doi:>9} {n_cited:>11}")
    totals['total'] += n_total
    totals['valid'] += n_valid
    totals['abstract'] += n_abstract
    totals['doi'] += n_doi
    totals['cited'] += n_cited

print(sep)
print(f"{'TOTAL':<22} {totals['total']:>9} {totals['valid']:>7} {totals['abstract']:>10} {totals['doi']:>9} {totals['cited']:>11}")
print()

# Validation error breakdown by source
print("=== VALIDATION ERROR BREAKDOWN BY SOURCE ===\n")
errors_df = report.errors_to_dataframe()
if len(errors_df) > 0:
    for source in presentation_sources:
        src_idx = set(raw_df[raw_df['source'] == source].index)
        src_errors = errors_df[errors_df['row_index'].isin(src_idx)]
        print(f"{source}: {len(src_errors)} errors")
        if len(src_errors) > 0:
            print(src_errors['field'].value_counts().to_string())
else:
    print("No validation errors found across all sources.")

=== DATASET COVERAGE STATS FOR PRESENTATION ===

Source                   Records   Valid   Abstract   Has DOI  Cited Art.
-------------------------------------------------------------------------
dryad                         47      37         46        47          44
zenodo                       114      67        114       114          59
semantic_scholar             254     192        246       156           0
-------------------------------------------------------------------------
TOTAL                        415     296        406       317         103

=== VALIDATION ERROR BREAKDOWN BY SOURCE ===

No validation errors found across all sources.


## 7. Export Validated Data

In [17]:
# valid_records_df: schema-valid rows filtered to valid_yn == 'yes'
# (computed in Section 6 above)
print(f"Records to export: {len(valid_records_df)}")
print(f"\nBy source:")
print(valid_records_df['source'].value_counts())
valid_records_df.head()

Records to export: 299

By source:
source
semantic_scholar    192
zenodo               67
dryad                37
referenced            3
Name: count, dtype: int64


,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,species,referred_dataset,time_series,multispecies,...,valid_yn,reason_not_valid,source,source_url,journal_url,pdf_url,is_oa,cited_article_doi,id,title
0,[genetic_analysis],None,NaN,NaN,NaN,NaN,[41 fish mock species],NaN,None,None,...,yes,None,zenodo,https://doi.org/10.5061/dryad.05qfttf0n,None,None,None,NaN,3,Fish mock community with 41 species from 13 or...
1,[presence-only],[administrative_units],0.2,2012-2017,2012.0,2017.0,[Tamias striatus],NaN,True,False,...,yes,None,zenodo,https://doi.org/10.5061/dryad.0rxwdbs2c,https://doi.org/10.5281/zenodo.5898699,None,True,https://doi.org/10.5281/zenodo.5898699,4,Exploration and diet specialization in eastern...
2,[genetic_analysis],None,NaN,2006-2007,2006.0,2007.0,[Glyptemys insculpta],NaN,None,False,...,yes,None,zenodo,https://doi.org/10.5061/dryad.121sk,https://doi.org/10.1093/jhered/esx103,https://academic.oup.com/jhered/article-pdf/10...,True,https://doi.org/10.1093/jhered/esx103,5,Data from: Paternity analysis of wood turtles ...
3,[other],None,NaN,NaN,NaN,NaN,[40 freshwater vertebrate and invertebrates],NaN,None,None,...,yes,None,zenodo,https://doi.org/10.5061/dryad.12jm63xtw,None,None,None,NaN,7,Data from: 60 specific eDNA qPCR assays to det...
4,[density],[sample],NaN,2008,2008.0,2008.0,"[raccoons, striped skunks]",NaN,None,False,...,yes,None,dryad,https://doi.org/10.5061/dryad.1771t,https://doi.org/10.1371/journal.pone.0128238,https://journals.plos.org/plosone/article/file...,True,https://doi.org/10.1371/journal.pone.0128238,9,Data from: Resampling method for applying dens...


In [18]:
# Export valid records (schema-valid + valid_yn='yes') to OUTPUT_FILE
valid_records_df.to_excel(OUTPUT_FILE, index=False)
print(f"Exported {len(valid_records_df)} validated records to {OUTPUT_FILE}")
print(f"\nSummary: {len(valid_records_df)} valid records from {len(raw_df)} total")
print(f"  Dryad:            {(valid_records_df['source'] == 'dryad').sum()}")
print(f"  Zenodo:           {(valid_records_df['source'] == 'zenodo').sum()}")
print(f"  Semantic Scholar: {(valid_records_df['source'] == 'semantic_scholar').sum()}")

print(f"\n=== ENRICHED FIELD COVERAGE IN EXPORT ===")
for col in ['source_url', 'cited_article_doi', 'journal_url', 'pdf_url', 'is_oa']:
    if col in valid_records_df.columns:
        n = valid_records_df[col].notna().sum()
        total = len(valid_records_df)
        print(f"  {col:<22}: {n:>4}/{total} ({100*n/total:.1f}%)")
    else:
        print(f"  {col:<22}: column missing")

Exported 299 validated records to data/dataset_092624_validated.xlsx

Summary: 299 valid records from 418 total
  Dryad:            37
  Zenodo:           67
  Semantic Scholar: 192

=== ENRICHED FIELD COVERAGE IN EXPORT ===
  source_url            :  279/299 (93.3%)
  cited_article_doi     :  250/299 (83.6%)
  journal_url           :   98/299 (32.8%)
  pdf_url               :  189/299 (63.2%)
  is_oa                 :  194/299 (64.9%)
